# Part D: Softmax classifier (15 points)

Dataset: **Digits** (pixels / 16, no standardization). Scores S = X̃W, P = softmax(S) row-wise, and
L(W) = -(1/N) Σ log P[i, y_i] + (λ/2)||W_1:||², where the bias row of W is not regularized.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits, load_breast_cancer, load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression          # only for the off-the-shelf tasks
from sklearn.metrics import confusion_matrix as sk_confusion  # only for checking my own version

# ---- load digits and split (protocol of Section 4.1) ----
data = load_digits()
X = data.data / 16.0
y = data.target
K = 10

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_tr, X_va, y_tr, y_va = train_test_split(X_tr, y_tr, test_size=0.25, random_state=42, stratify=y_tr)
print(X_tr.shape, X_va.shape, X_te.shape)

## Helpers from Parts A (copied here so this part runs on its own)

In [ ]:
def add_bias(X):
    return np.hstack([np.ones((X.shape[0], 1)), X])

class Standardizer:
    def fit(self, X):
        self.mu = X.mean(axis=0)
        self.sigma = X.std(axis=0)
        self.sigma[self.sigma == 0] = 1      # constant columns: use sigma = 1
        return self
    def transform(self, X):
        return (X - self.mu) / self.sigma

def confusion_matrix(y, pred, K):
    # rows = true class, columns = predicted class
    cm = np.zeros((K, K), dtype=int)
    for t, p in zip(y, pred):
        cm[t, p] += 1
    return cm

def per_class_metrics(cm):
    tp = np.diag(cm).astype(float)
    pred_count = cm.sum(axis=0)
    true_count = cm.sum(axis=1)
    precision = np.where(pred_count > 0, tp / np.maximum(pred_count, 1), 0.0)  # precision = 0 if class never predicted
    recall = np.where(true_count > 0, tp / np.maximum(true_count, 1), 0.0)
    f1 = np.where(precision + recall > 0, 2 * precision * recall / np.maximum(precision + recall, 1e-12), 0.0)
    return precision, recall, f1

def grad_check(f, grad, w, n_entries=100, eps=1e-5):
    # central differences on (at most) n_entries random entries of w
    g_an = grad(w)
    flat_idx = rng.choice(w.size, size=min(n_entries, w.size), replace=False)
    g_an_sel = np.zeros(len(flat_idx))
    g_num_sel = np.zeros(len(flat_idx))
    for k, idx in enumerate(flat_idx):
        w_plus = w.copy();  w_plus.flat[idx] += eps
        w_minus = w.copy(); w_minus.flat[idx] -= eps
        g_num_sel[k] = (f(w_plus) - f(w_minus)) / (2 * eps)
        g_an_sel[k] = g_an.flat[idx]
    num = np.linalg.norm(g_an_sel - g_num_sel)
    den = max(np.linalg.norm(g_an_sel), np.linalg.norm(g_num_sel), 1e-12)
    return num / den

rng = np.random.default_rng(42)
Xb_tr, Xb_va, Xb_te = add_bias(X_tr), add_bias(X_va), add_bias(X_te)

## Task 17: Stable softmax and gradient

**Derivation.** For one sample, ℓ = -log P[y] = -S[y] + log Σ_j exp(S[j]). Then

∂ℓ/∂S[k] = -1[k = y] + exp(S[k]) / Σ_j exp(S[j]) = P[k] - Y[k].

Since S = X̃W, ∂S[i,k]/∂W[:,k] = x̃_i, so averaging over the samples gives (1/N) X̃ᵀ(P − Y).
The regularizer (λ/2)||W_1:||² has gradient λW_1:, and the bias row is not penalized, so its gradient is 0.
Together: **∂L/∂W = (1/N) X̃ᵀ(P − Y) + λ[0; W_1:]**.

**Stability proof (one line).** softmax(s + c·1)_k = e^{s_k + c} / Σ_j e^{s_j + c} = e^c e^{s_k} / (e^c Σ_j e^{s_j}) = softmax(s)_k, so we can subtract max(s) from each row and nothing changes, but exp never overflows.

In [ ]:
def softmax(S):
    S = S - S.max(axis=1, keepdims=True)      # max-subtraction trick
    E = np.exp(S)
    return E / E.sum(axis=1, keepdims=True)

def data_loss(W, Xb, y):
    # cross-entropy through log-sum-exp (never np.log(softmax(...)))
    S = Xb @ W
    m = S.max(axis=1, keepdims=True)
    lse = m[:, 0] + np.log(np.exp(S - m).sum(axis=1))
    return np.mean(lse - S[np.arange(len(y)), y])

def full_loss(W, Xb, y, lam):
    return data_loss(W, Xb, y) + lam / 2 * np.sum(W[1:] ** 2)

def gradient(W, Xb, y, lam):
    P = softmax(Xb @ W)
    Y = np.zeros_like(P)
    Y[np.arange(len(y)), y] = 1
    g = Xb.T @ (P - Y) / len(y)
    g[1:] += lam * W[1:]                      # bias row is not regularized
    return g

def accuracy(W, Xb, y):
    return np.mean(np.argmax(Xb @ W, axis=1) == y)

# the [1000, 1001, 1002] example from Section 3.3
print(softmax(np.array([[1000., 1001., 1002.]])))     # expected [0.0900, 0.2447, 0.6652]

In [ ]:
# at W = 0 every class has probability 1/K, so the loss is -log(1/10) = ln 10
W0 = np.zeros((Xb_tr.shape[1], K))
print("loss at W=0:", data_loss(W0, Xb_tr, y_tr), " ln(10) =", np.log(10))

# gradient check at a random point, lambda = 0.1, 100 random entries
lam = 0.1
W_rand = rng.normal(0, 0.1, size=W0.shape)
err = grad_check(lambda W: full_loss(W, Xb_tr, y_tr, lam),
                 lambda W: gradient(W, Xb_tr, y_tr, lam), W_rand)
print("relative error:", err)

**Interpretation.** With W = 0 all scores are 0, so every class gets probability 1/10 and the loss is −log(1/10) = ln 10 ≈ 2.3026 for every sample, which is what we got. The gradient check gives a relative error far below 1e-6, so the analytic gradient is correct.

## Task 18: Training and model selection

Mini-batch GD with B = 64 for 100 epochs. Each epoch I reshuffle the training set and keep the last partial batch.
The reported loss is the data term only (no regularizer).

In [ ]:
def train_softmax(Xb_tr, y_tr, Xb_va, y_va, eta, lam, epochs, B, n_classes=10, seed=42):
    rng_local = np.random.default_rng(seed)
    N = Xb_tr.shape[0]
    W = np.zeros((Xb_tr.shape[1], n_classes))
    hist = {"tr_loss": [], "va_loss": [], "tr_acc": [], "va_acc": []}
    for ep in range(epochs):
        perm = rng_local.permutation(N)                 # reshuffle every epoch
        for start in range(0, N, B):                    # last partial batch is kept
            idx = perm[start:start + B]
            W -= eta * gradient(W, Xb_tr[idx], y_tr[idx], lam)
        hist["tr_loss"].append(data_loss(W, Xb_tr, y_tr))
        hist["va_loss"].append(data_loss(W, Xb_va, y_va))
        hist["tr_acc"].append(accuracy(W, Xb_tr, y_tr))
        hist["va_acc"].append(accuracy(W, Xb_va, y_va))
    return W, hist

### (a) λ = 0, learning rates

In [ ]:
etas = [0.01, 0.1, 0.5, 1.0]
runs_a = {}
for eta in etas:
    runs_a[eta] = train_softmax(Xb_tr, y_tr, Xb_va, y_va, eta, 0.0, 100, 64)

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for eta in etas:
    h = runs_a[eta][1]
    ax[0].plot(h["tr_loss"], label=f"eta={eta}")
    ax[1].plot(h["va_loss"], label=f"eta={eta}")
    ax[2].plot(h["va_acc"], label=f"eta={eta}")
ax[0].set_title("Train loss vs epoch (lambda=0)"); ax[1].set_title("Validation loss vs epoch (lambda=0)")
ax[2].set_title("Validation accuracy vs epoch (lambda=0)")
for a, yl in zip(ax, ["loss", "loss", "accuracy"]):
    a.set_xlabel("epoch"); a.set_ylabel(yl); a.legend()
plt.tight_layout(); plt.show()

print("eta    train loss  val loss  train acc  val acc")
for eta in etas:
    h = runs_a[eta][1]
    print(f"{eta:<6} {h['tr_loss'][-1]:.4f}      {h['va_loss'][-1]:.4f}    {h['tr_acc'][-1]:.4f}     {h['va_acc'][-1]:.4f}")

eta_star = min(etas, key=lambda e: runs_a[e][1]["va_loss"][-1])
print("eta* =", eta_star)

### (b) fix η*, vary λ

In [ ]:
lams = [0, 1e-4, 1e-3, 1e-2, 1e-1]
runs_b = {}
for lam in lams:
    runs_b[lam] = train_softmax(Xb_tr, y_tr, Xb_va, y_va, eta_star, lam, 100, 64)

print("lambda   train acc  val acc  val loss")
for lam in lams:
    h = runs_b[lam][1]
    print(f"{lam:<8} {h['tr_acc'][-1]:.4f}     {h['va_acc'][-1]:.4f}   {h['va_loss'][-1]:.4f}")

# best validation accuracy, ties -> larger lambda
best_acc = max(runs_b[l][1]["va_acc"][-1] for l in lams)
lam_star = max(l for l in lams if runs_b[l][1]["va_acc"][-1] == best_acc)
W_star = runs_b[lam_star][0]
print("lambda* =", lam_star)
print("TEST accuracy (reported once):", accuracy(W_star, Xb_te, y_te))

**Interpretation.** (a) The smallest rate η = 0.01 is clearly still under-trained after 100 epochs (loss still high, accuracy lower), while the larger rates reach a low loss quickly. I picked η* as the one with the lowest final validation loss. (b) Small λ values give almost the same validation accuracy, since the data is easy and regularization matters little; λ = 0.1 is too strong and hurts accuracy because it pulls the weights too close to zero. The test accuracy was only computed once, after choosing η* and λ* on validation data.

## Task 19: Off-the-shelf comparison

scikit-learn's `LogisticRegression` minimizes C·Σloss + ½||W||², which matches my objective with C = 1/(λ·N_train). For λ = 0 I use no penalty.

In [ ]:
N_train = X_tr.shape[0]

def sk_model(C=None, lam=None, N=N_train):
    if lam is not None and lam == 0:
        return LogisticRegression(penalty=None, max_iter=10000)
    if lam is not None:
        C = 1.0 / (lam * N)
    return LogisticRegression(C=C, max_iter=10000)

print("lambda   mine val  mine test | sk val  sk test | agreement on test")
sk_preds = {}
for lam in lams:
    W = runs_b[lam][0]
    m = sk_model(lam=lam).fit(X_tr, y_tr)
    sk_preds[lam] = m.predict(X_te)
    my_pred = np.argmax(Xb_te @ W, axis=1)
    print(f"{lam:<8} {accuracy(W, Xb_va, y_va):.4f}    {accuracy(W, Xb_te, y_te):.4f}    | "
          f"{m.score(X_va, y_va):.4f}  {m.score(X_te, y_te):.4f}  | {np.mean(my_pred == sk_preds[lam]):.4f}")

my_pred_star = np.argmax(Xb_te @ W_star, axis=1)
print("agreement at lambda*:", np.mean(my_pred_star == sk_preds[lam_star]))

In [ ]:
print("C      val acc   test acc")
for C in [0.1, 1, 10, 100]:
    m = sk_model(C=C).fit(X_tr, y_tr)
    print(f"{C:<6} {m.score(X_va, y_va):.4f}    {m.score(X_te, y_te):.4f}")

**Interpretation.** My from-scratch classifier and scikit-learn give very similar validation and test accuracies for every λ, and at λ* their test predictions agree on at least 97% of the images. Small differences come from the optimizers: I stop after 100 epochs of mini-batch GD, while scikit-learn's lbfgs runs until convergence. Increasing C (weaker regularization) in the sweep gives similar accuracy for the larger values.

## Task 20: Error analysis (λ = λ*, my model)

In [ ]:
pred_te = np.argmax(Xb_te @ W_star, axis=1)
cm = confusion_matrix(y_te, pred_te, K)
print("matches sklearn:", np.array_equal(cm, sk_confusion(y_te, pred_te, labels=range(K))))

plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap="Blues")
plt.colorbar()
for i in range(K):
    for j in range(K):
        plt.text(j, i, cm[i, j], ha="center", va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.xticks(range(K)); plt.yticks(range(K))
plt.xlabel("predicted class"); plt.ylabel("true class"); plt.title("Test confusion matrix (digits)")
plt.show()

prec, rec, f1 = per_class_metrics(cm)
print("digit  precision  recall  F1")
for k in range(K):
    print(f"{k}      {prec[k]:.3f}      {rec[k]:.3f}   {f1[k]:.3f}")

In [ ]:
# three most confused (true -> predicted) pairs
off = cm.copy()
np.fill_diagonal(off, 0)
order = np.argsort(off, axis=None)[::-1][:3]
print("Most confused pairs:")
for o in order:
    t, p = divmod(o, K)
    print(f"  true {t} -> predicted {p}: {off[t, p]} times")

# six misclassified test digits
wrong = np.where(pred_te != y_te)[0][:6]
fig, axes = plt.subplots(1, len(wrong), figsize=(2 * len(wrong), 2.5))
axes = np.atleast_1d(axes)
for a, i in zip(axes, wrong):
    a.imshow(X_te[i].reshape(8, 8), cmap="gray_r")
    a.set_title(f"{y_te[i]} / {pred_te[i]}")
    a.axis("off")
plt.suptitle("Misclassified test digits (true / predicted)")
plt.show()

In [ ]:
# each class's 64 weights (without the bias) as an 8x8 image
fig, axes = plt.subplots(1, 10, figsize=(20, 2.5))
for k in range(K):
    axes[k].imshow(W_star[1:, k].reshape(8, 8), cmap="coolwarm")
    axes[k].set_title(f"digit {k}")
    axes[k].axis("off")
plt.suptitle("Weights of each class")
plt.show()

**Analysis (from my run).** The hardest digits are 1 and 8 (lowest recall and F1), followed by 9; the most confused pairs are 8 → 1, 1 → 9 and 8 → 7. Digits like 0, 2 and 3 are almost perfect. This makes sense for a linear model on raw pixels because each class only gets one template of weights (see the weight images, which look like blurry versions of the digits). A template can't adapt to different handwriting styles, and two digits that share many ink pixels get similar scores. Different ways of writing the same digit (slanted, thick, thin) also shift the pixels away from the template.

## Task 21: Softmax with K = 2 is logistic regression

**Proof.** For a two-class softmax with scores s₀ = w₀·x̃ and s₁ = w₁·x̃:

P(y=1|x) = e^{s₁} / (e^{s₀} + e^{s₁}) = 1 / (1 + e^{s₀ − s₁}) = σ(s₁ − s₀) = σ((w₁ − w₀)ᵀx̃).

So the two-class softmax has exactly the same form as logistic regression with weight vector w = w₁ − w₀ (only the difference of the weight vectors matters).

In [ ]:
bc = load_breast_cancer()
Xc = bc.data
yc = 1 - bc.target                                # malignant = 1
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.2, random_state=42, stratify=yc)

sc = Standardizer().fit(Xc_tr)                    # training statistics only
Xc_te_b = add_bias(sc.transform(Xc_te))           # 114 x 31

W2 = rng.normal(0, 1, size=(31, 2))               # random 31x2 weight matrix
P2 = softmax(Xc_te_b @ W2)

sigmoid = lambda z: 1 / (1 + np.exp(-z))
p_log = sigmoid(Xc_te_b @ (W2[:, 1] - W2[:, 0]))

print("max |softmax P(y=1) - sigmoid((w1-w0).x)| =", np.max(np.abs(P2[:, 1] - p_log)))
print("max |softmax P(y=0) - (1 - sigmoid)|      =", np.max(np.abs(P2[:, 0] - (1 - p_log))))

**Interpretation.** The difference is around 1e-16, well below 1e-12, so the two probabilities are the same up to floating point error. This confirms the proof.

## Task 22: Seeing the boundaries (Iris)

Features: petal length and petal width, standardized with training statistics. Full batch, η = 0.5, 2000 epochs, λ ∈ {0, 1}.

In [ ]:
iris = load_iris()
Xi = iris.data[:, 2:4]
yi = iris.target
Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(Xi, yi, test_size=0.2, random_state=42, stratify=yi)
Xi_tr, Xi_va, yi_tr, yi_va = train_test_split(Xi_tr, yi_tr, test_size=0.25, random_state=42, stratify=yi_tr)

sci = Standardizer().fit(Xi_tr)
Xi_tr_s, Xi_va_s, Xi_te_s = sci.transform(Xi_tr), sci.transform(Xi_va), sci.transform(Xi_te)
Xib_tr, Xib_va = add_bias(Xi_tr_s), add_bias(Xi_va_s)

def plot_decision_regions(predict, X, y, ax, title):
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
    grid = np.c_[xx.ravel(), yy.ravel()]
    zz = predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, zz, levels=[-0.5, 0.5, 1.5, 2.5], alpha=0.3, cmap="viridis")
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="viridis", edgecolor="k")
    ax.set_xlabel("petal length (standardized)"); ax.set_ylabel("petal width (standardized)")
    ax.set_title(title)

N_iris = Xi_tr_s.shape[0]
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for col, lam in enumerate([0, 1]):
    # my softmax (full batch: B = N)
    W_i, _ = train_softmax(Xib_tr, yi_tr, Xib_va, yi_va, 0.5, lam, 2000, N_iris, n_classes=3)
    plot_decision_regions(lambda G: np.argmax(add_bias(G) @ W_i, axis=1), Xi_tr_s, yi_tr,
                          axes[0, col], f"My softmax, lambda={lam}")
    # scikit-learn at the matching C
    m = sk_model(lam=lam, N=N_iris).fit(Xi_tr_s, yi_tr)
    plot_decision_regions(m.predict, Xi_tr_s, yi_tr, axes[1, col],
                          f"scikit-learn, lambda={lam}" + ("" if lam == 0 else f" (C={1/(lam*N_iris):.4f})"))
    print(f"lambda={lam}: train acc mine {accuracy(W_i, Xib_tr, yi_tr):.3f}, sklearn {m.score(Xi_tr_s, yi_tr):.3f}")
plt.tight_layout(); plt.show()

**Interpretation.** With three classes and two features, the decision regions are three convex regions separated by straight lines meeting at a point. For λ = 0 the boundaries are sharper because the weights can grow freely; with λ = 1 the weights are shrunk, so the probabilities are softer and the boundaries move (the versicolor/virginica boundary especially), and more points near it are misclassified. My regions look the same as scikit-learn's for the matching C, which shows the from-scratch implementation is right.